In [1]:
import pyspark
import pyspark.sql.functions as F
import math

In [2]:
spark = pyspark.sql.SparkSession.builder.appName("bronze").config("spark.driver.maxResultSize", "16g") \
    .config("spark.driver.memory", "32g").getOrCreate()
spark.conf.set("spark.sql.caseSensitive", "true")

# Print the SparkSession details to verify
print(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/15 17:15:20 WARN Utils: Your hostname, rocinante, resolves to a loopback address: 127.0.1.1; using 192.168.0.226 instead (on interface enp37s0f0)
26/10/15 17:15:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/15 17:15:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df = spark.read.parquet("../silver/idnet100k_no_dupe.parquet")

In [4]:
df.show()

+--------------------+----------+-----+---------+-----------+---------+------+----------+-------------+--------------------+--------------------+--------+----------+----------+-----------------+--------------------+----------------+----------------+----------+
|             address|  birthday|class|ethnicity|expire_date|eye_color|gender|hair_color|height_inches|                  id|           imageData|is_donor|is_veteran|issue_date|   license_number|                name|     person_name|prelabeled_fraud|weight_lbs|
+--------------------+----------+-----+---------+-----------+---------+------+----------+-------------+--------------------+--------------------+--------+----------+----------+-----------------+--------------------+----------------+----------------+----------+
|551 Montezuma Str...|12/04/2000|    B|    white| 10/05/2025|      BRO|     F|       BRO|           68|az-generated.phot...|iVBORw0KGgoAAAANS...|   false|      true|10/05/2020|        D65357893|az-generated.phot...|  

In [5]:
df.select('license_number').count()

106060

In [6]:
df.select('license_number').distinct().count()

63634

In [7]:
spark.sql('select count(*) from {df} where license_number is null', df=df).show()

+--------+
|count(1)|
+--------+
|       0|
+--------+



In [8]:
lic_df = df.select('license_number')

In [9]:
new_doc_number = 5

alt_docs = lic_df


for x in range(new_doc_number-1):
    alt_docs = alt_docs.unionAll(lic_df)

In [10]:
alt_docs.show()

+-----------------+
|   license_number|
+-----------------+
|        D65357893|
|         I6272255|
|          6753467|
|          6156388|
|          2812325|
|       4763467857|
|       1495225605|
|       80 697 139|
|        267914704|
|        341153898|
|S082-1115-3116-99|
|S280-3023-1154-46|
|          W217951|
|          W716545|
|          W831803|
|          W447800|
|        D92730548|
|         I8225264|
|          6072502|
|          7697319|
+-----------------+
only showing top 20 rows


In [11]:
alt_docs.select('license_number').count()

530300

In [12]:
alt_docs.select('license_number').distinct().count()

63634

In [13]:
#labeled_training_set.write.mode("overwrite").parquet("./silver/labeled_training_set_no_tensor_10k.parquet")

In [14]:
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DecimalType

In [15]:
schema = StructType([
    StructField("license_number", StringType(), True),
    StructField("plate_number", StringType(), True),
    StructField("photo_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("toll_id", StringType(), True),
    StructField("speed", IntegerType(), True),
    StructField("pass_device_type", StringType(), True),
    StructField("driver_hair_color", StringType(), True),
    StructField("driver_eye_color", StringType(), True),
    StructField("driver_weight", StringType(), True),
    StructField("toll_amount_usd", DecimalType(8,2), True),
])

In [16]:
from faker import Faker

In [17]:
fake = Faker()

In [18]:
from datetime import datetime

@F.udf(StringType())
def generate_plate_num():
    return fake.license_plate()

@F.udf(StringType())
def generate_photo_id():
    return fake.uuid4()

@F.udf(StringType())
def generate_timestamp ():
    fake_date = fake.date_time_between_dates(datetime(2024,1,1), datetime(2024,12,31))
    return fake_date.strftime("%Y-%m-%d %H:%M:%S")

@F.udf(StringType())
def generate_toll_id():
    return fake.pyint(1,100000)


@F.udf(StringType())
def generate_speed():
    return fake.pyint(25,100)

@F.udf(StringType())
def generate_pass_device_type():
    return fake.pyint(1,4)

@F.udf(StringType())
def generate_driver_hair_color():
    hair_colors = ['BRO','BLK', 'BLN']
    return fake.random_element(hair_colors)

@F.udf(StringType())
def generate_driver_eye_color():
    eye_colors = ['BRO','BLU', 'GRY']
    return fake.random_element(eye_colors)

@F.udf(StringType())
def generate_driver_weight():
    return fake.pyint(70,300)


@F.udf(StringType())
def generate_toll_amount_usd():
    amount = fake.pydecimal(min_value=5,max_value=90, right_digits=2)
    return amount



In [19]:
alt_docs = alt_docs.withColumn('plate_number', generate_plate_num())
alt_docs = alt_docs.withColumn('toll_id', generate_toll_id())
alt_docs = alt_docs.withColumn('timestamp', generate_timestamp())
alt_docs = alt_docs.withColumn('photo_id', generate_photo_id())
alt_docs = alt_docs.withColumn('speed', generate_speed())
alt_docs = alt_docs.withColumn('pass_device_type', generate_pass_device_type())
alt_docs = alt_docs.withColumn('driver_hair_color', generate_driver_hair_color())
alt_docs = alt_docs.withColumn('driver_eye_color', generate_driver_eye_color())
alt_docs = alt_docs.withColumn('driver_weight', generate_driver_weight())
alt_docs = alt_docs.withColumn('toll_amount_usd', generate_toll_amount_usd())


In [20]:
# To reduce by 35%, sample and keep 65% (0.65) of the data
alt_docs_reduced = alt_docs.sample(withReplacement=False, fraction=0.65, seed=42)


In [21]:
alt_docs_reduced.show()

+-----------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+
|   license_number|plate_number|toll_id|          timestamp|            photo_id|speed|pass_device_type|driver_hair_color|driver_eye_color|driver_weight|toll_amount_usd|
+-----------------+------------+-------+-------------------+--------------------+-----+----------------+-----------------+----------------+-------------+---------------+
|        D65357893|    2K 37614|  27202|2024-03-16 17:57:48|ee7893e8-dcec-40d...|   51|               2|              BRO|             BRO|          123|          31.52|
|         I6272255|     HKS 545|  41043|2024-11-10 23:46:16|759a2c96-3181-4fe...|   65|               3|              BLK|             BLU|          150|          29.76|
|          6156388|     YKS-625|  25347|2024-03-11 14:00:12|eb7de36a-50c8-4f2...|   49|               2|              BRO|             BRO|          2

In [22]:
print(fake.license_plate())

2K 37614


In [23]:
for x in range(5):
    print(fake.license_plate())

HKS 545
8725 XK
YKS-625
7Y-VLVN
484 JGJ


In [24]:
total_rows = alt_docs_reduced.count()


num_partitions = math.ceil(total_rows / 10)


alt_docs_reduced = alt_docs_reduced.repartition(num_partitions)

In [25]:
alt_docs_reduced.write.mode("overwrite").parquet("../silver/alt_docs_x5_no_dupe.parquet")

In [26]:
alt_docs_reduced.count()

344209